In [2]:
import re
import pandas as pd
import numpy as np

# 1. 데이터 로드 (Terminal에서 wget으로 다운로드한 파일)
# data = pd.read_csv('~/work/transformer_chatbot/data/ChatbotData.csv')
# 실습을 위해 가상 데이터로 예시를 듭니다.
data = pd.DataFrame({
    'Q': ['12시 전에는 잘 거야', '너의 이름은 뭐니?', '오늘 날씨 좋다'],
    'A': ['건강을 위해 일찍 주무세요.', '저는 AI 챗봇입니다.', '산책하기 좋은 날씨네요.']
})

# 2. 전처리 함수 (구두점 분리 및 특수문자 제거)
def preprocess_sentence(sentence):
    sentence = sentence.lower().strip()
    # 구두점 양옆에 공백 추가
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r'[" "]+', " ", sentence)
    # 한글, 영어, 숫자, 구두점을 제외한 문자 제거
    sentence = re.sub(r"[^ㄱ-ㅎㅏ-ㅣ가-힣a-zA-Z0-9?.!,]+", " ", sentence)
    return sentence.strip()

data['Q'] = data['Q'].apply(preprocess_sentence)
data['A'] = data['A'].apply(preprocess_sentence)

In [ ]:
import sentencepiece as spm
import torch
from torch.utils.data import Dataset, DataLoader

# 1. SentencePiece 학습을 위한 텍스트 파일 저장
with open('output_ChatbotData.csv', 'w', encoding='utf-8') as f:
    for text in pd.concat([data['Q'], data['A']]):
        f.write(text + '\n')

# 2. SentencePiece 모델 학습 (스페셜 토큰 포함)
# spm.SentencePieceTrainer.Train(
#     '--input=work\songys_chatbot\ChatbotData.csv --model_prefix=chatbot_spm '
#     '--vocab_size=1000 --model_type=bpe '
#     '--pad_id=0 --bos_id=1 --eos_id=2 --unk_id=3 '
#     '--pad_piece=<PAD> --bos_piece=<START> --eos_piece=<END> --unk_piece=<UNK>'
# )
# SentencePieceTrainer 코드에서 vocab_size를 낮춰줍니다.
spm.SentencePieceTrainer.Train(
    '--input=work\songys_chatbot\ChatbotData.csv --model_prefix=chatbot_spm '
    '--vocab_size=100 --model_type=bpe ' # 1000에서 100으로 변경
    '--pad_id=0 --bos_id=1 --eos_id=2 --unk_id=3 '
    '--pad_piece=<PAD> --bos_piece=<START> --eos_piece=<END> --unk_piece=<UNK>'
)
sp = spm.SentencePieceProcessor()
sp.Load('chatbot_spm.model')

MAX_LENGTH = 15

# 3. 데이터셋 클래스 정의 (교사 강요 적용)
class ChatbotDataset(Dataset):
    def __init__(self, df, sp, max_length):
        self.inputs = []
        self.dec_inputs = []
        self.labels = []
        
        for q, a in zip(df['Q'], df['A']):
            # 인코더 입력: 질문
            enc_input = [sp.bos_id()] + sp.EncodeAsIds(q) + [sp.eos_id()]
            # 디코더 입력/레이블을 위한 답변 토큰화
            dec_seq = [sp.bos_id()] + sp.EncodeAsIds(a) + [sp.eos_id()]
            
            # 교사 강요 분리: 입력은 마지막 토큰 제외, 레이블은 첫 토큰 제외
            dec_input = dec_seq[:-1]
            label = dec_seq[1:]
            
            # 패딩 처리
            enc_input = enc_input[:max_length] + [sp.pad_id()] * (max_length - len(enc_input))
            dec_input = dec_input[:max_length] + [sp.pad_id()] * (max_length - len(dec_input))
            label = label[:max_length] + [sp.pad_id()] * (max_length - len(label))
            
            self.inputs.append(enc_input)
            self.dec_inputs.append(dec_input)
            self.labels.append(label)
            
    def __len__(self):
        return len(self.inputs)
    
    def __getitem__(self, idx):
        return (torch.tensor(self.inputs[idx]), 
                torch.tensor(self.dec_inputs[idx]), 
                torch.tensor(self.labels[idx]))

dataset = ChatbotDataset(data, sp, MAX_LENGTH)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

<>:6: SyntaxWarning: invalid escape sequence '\s'
<>:19: SyntaxWarning: invalid escape sequence '\s'
<>:6: SyntaxWarning: invalid escape sequence '\s'
<>:19: SyntaxWarning: invalid escape sequence '\s'
C:\Users\H11\AppData\Local\Temp\ipykernel_28864\3519190255.py:6: SyntaxWarning: invalid escape sequence '\s'
  with open('work\songys_chatbot\ChatbotData.csv', 'w', encoding='utf-8') as f:
C:\Users\H11\AppData\Local\Temp\ipykernel_28864\3519190255.py:19: SyntaxWarning: invalid escape sequence '\s'
  '--input=work\songys_chatbot\ChatbotData.csv --model_prefix=chatbot_spm '


In [6]:
import torch.nn as nn
import math

class TransformerChatbot(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, max_len):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        
        # 포지셔널 인코딩 (간단한 학습형으로 구현)
        self.pos_embedding = nn.Embedding(max_len, d_model)
        
        # 파이토치 내장 트랜스포머
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead, 
            num_encoder_layers=num_layers, num_decoder_layers=num_layers,
            dim_feedforward=2048, batch_first=True
        )
        self.fc_out = nn.Linear(d_model, vocab_size)
        
    def generate_square_subsequent_mask(self, sz, device):
        mask = torch.triu(torch.ones(sz, sz, device=device)) == 1
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask

    def forward(self, src, tgt, src_pad_idx, tgt_pad_idx):
        src_seq_len = src.shape[1]
        tgt_seq_len = tgt.shape[1]
        device = src.device
        
        # 포지션 임베딩 더하기
        src_pos = torch.arange(0, src_seq_len, device=device).unsqueeze(0)
        tgt_pos = torch.arange(0, tgt_seq_len, device=device).unsqueeze(0)
        
        src_emb = self.embedding(src) * math.sqrt(self.d_model) + self.pos_embedding(src_pos)
        tgt_emb = self.embedding(tgt) * math.sqrt(self.d_model) + self.pos_embedding(tgt_pos)
        
        # 패딩 마스크 및 디코더 룩어헤드 마스크 생성
        src_key_padding_mask = (src == src_pad_idx)
        tgt_key_padding_mask = (tgt == tgt_pad_idx)
        tgt_mask = self.generate_square_subsequent_mask(tgt_seq_len, device)
        
        # 트랜스포머 통과
        out = self.transformer(
            src_emb, tgt_emb, 
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask
        )
        return self.fc_out(out)

# 모델 생성 (d_model=512, nhead=8, num_layers=6)
VOCAB_SIZE = sp.GetPieceSize()
model = TransformerChatbot(vocab_size=VOCAB_SIZE, d_model=512, nhead=8, num_layers=6, max_len=MAX_LENGTH)

In [ ]:
def evaluate(sentence, model, sp, max_length=MAX_LENGTH):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()
    
    # 1. 입력 문장 전처리 및 토큰화
    sentence = preprocess_sentence(sentence)
    enc_input = [sp.bos_id()] + sp.EncodeAsIds(sentence) + [sp.eos_id()]
    enc_input = enc_input[:max_length] + [sp.pad_id()] * (max_length - len(enc_input))
    enc_input = torch.tensor(enc_input, dtype=torch.long).unsqueeze(0).to(device)
    
    # 2. 디코더 입력 초기화 (<START> 토큰으로 시작)
    decoder_input = torch.tensor([sp.bos_id()], dtype=torch.long).unsqueeze(0).to(device)
    
    # 3. 예측 루프
    for i in range(max_length):
        with torch.no_grad():
            predictions = model(enc_input, decoder_input, sp.pad_id(), sp.pad_id())
        
        # 마지막 시점의 예측 끄집어내기
        prediction = predictions[:, -1, :]
        predicted_id = torch.argmax(prediction, dim=-1).item()
        
        # <END> 토큰이 나오면 멈춤
        if predicted_id == sp.eos_id():
            break
            
        # 예측된 토큰을 디코더 입력에 추가(Append)
        decoder_input = torch.cat([decoder_input, torch.tensor([[predicted_id]]).to(device)], dim=1)
        
    # 4. ID 시퀀스를 텍스트로 복원
    predicted_ids = decoder_input.squeeze().tolist()[1:] # <START> 제외
    return sp.DecodeIds(predicted_ids)

# 실행 테스트 (학습 전이라 무작위 답변이 나옵니다)
print("로봇:", evaluate("오늘 날씨 좋다", model, sp))

c:\Users\H11\anaconda3\Lib\site-packages\torch\nn\modules\transformer.py:529: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\NestedTensorImpl.cpp:182.)
  output = torch._nested_tensor_from_mask(
c:\Users\H11\anaconda3\Lib\site-packages\torch\nn\modules\activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


로봇: 는 전 챗날책하기 이 잘날책하기 이날책하기 이 잘날


c:\Users\H11\anaconda3\Lib\site-packages\torch\nn\modules\activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


: 